In [ ]:
"""
analysis.py
-----------
Experiments and plots comparing smoothing functions for the
projectile contact problem.

Experiments:
  1. Smoothing functions and their derivatives (visual comparison)
  2. Loss landscape L_kappa(theta) for each smoothing
  3. Gradient field: FD vs FoG
  4. Convergence of gradient descent
  5. Smoothing bias ||theta*_kappa - theta*|| vs kappa
  6. ZoG vs FoG comparison at a single theta
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import os

from smoothing import (get_smoothing, SMOOTHING_NAMES, SMOOTHING_LABELS,
                       SMOOTHING_COLORS, SMOOTHING_LS)
from simulator import (Params, simulate, loss, grad_fd, grad_fog,
                       grad_zog, analytical_optimum)

OUT = '/home/claude/figures'
os.makedirs(OUT, exist_ok=True)

PARAMS = Params()
KAPPA  = 300.0   # default stiffness (as in Schwarke et al.)

# Default smoothing names to compare (exclude hard for most plots)
COMPARE = ['sigmoid', 'erf', 'smoothstep', 'sigmoid_mass']


# ============================================================
# Utilities
# ============================================================

def get_fn(name):
    return get_smoothing(name, mass=PARAMS.m)


def theta_grid(n=60, lo=-3.5, hi=3.5):
    t1 = np.linspace(lo, hi, n)
    t2 = np.linspace(lo, hi, n)
    return np.meshgrid(t1, t2)


# ============================================================
# Experiment 1: Smoothing functions and their derivatives
# ============================================================

def plot_smoothing_functions(kappa=KAPPA):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    d_vals = np.linspace(-0.05, 0.05, 500)

    for name in COMPARE + ['hard']:
        sigma_fn, sigma_prime_fn = get_fn(name)
        lbl   = SMOOTHING_LABELS[name]
        col   = SMOOTHING_COLORS[name]
        ls    = SMOOTHING_LS[name]

        axes[0].plot(d_vals, sigma_fn(d_vals, kappa),
                     label=lbl, color=col, ls=ls, lw=2)
        axes[1].plot(d_vals, sigma_prime_fn(d_vals, kappa),
                     label=lbl, color=col, ls=ls, lw=2)

    for ax in axes:
        ax.axvline(0, color='k', lw=0.7, ls=':')
        ax.set_xlabel(r'Penetration depth $d = -q_z^M$ [m]')
    axes[0].set_ylabel(r'$\sigma(d, \kappa)$')
    axes[1].set_ylabel(r"$\sigma'(d, \kappa)$")
    axes[0].set_title(r'Smoothing functions ($\kappa = {}$)'.format(int(kappa)))
    axes[1].set_title(r'Derivatives (gradient signal)')
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    path = f'{OUT}/smoothing_functions.pdf'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')


# ============================================================
# Experiment 2: Loss landscape
# ============================================================

def compute_loss_landscape(sigma_fn, kappa, n=40):
    T1, T2 = theta_grid(n=n, lo=-3.0, hi=3.0)
    L = np.zeros_like(T1)
    for i in range(n):
        for j in range(n):
            theta = np.array([T1[i, j], T2[i, j]])
            L[i, j] = loss(theta, sigma_fn, kappa, PARAMS)
    return T1, T2, L


def plot_loss_landscapes(kappa=KAPPA, n=35):
    theta_star = analytical_optimum(PARAMS)
    fig, axes = plt.subplots(1, len(COMPARE), figsize=(14, 3.5),
                             sharey=True)

    for ax, name in zip(axes, COMPARE):
        sigma_fn, _ = get_fn(name)
        T1, T2, L = compute_loss_landscape(sigma_fn, kappa, n=n)
        vmax = np.percentile(L, 95)
        cf = ax.contourf(T1, T2, np.clip(L, 0, vmax),
                         levels=20, cmap='viridis')
        ax.contour(T1, T2, L, levels=10, colors='white',
                   linewidths=0.5, alpha=0.4)
        ax.scatter(*theta_star, c='red', s=80, zorder=5,
                   label=r'$\theta^*$ analytical')
        # Find numerical minimum
        idx = np.unravel_index(np.argmin(L), L.shape)
        ax.scatter(T1[idx], T2[idx], c='yellow', marker='*',
                   s=120, zorder=5, label=r'$\theta^*_\kappa$ numerical')
        ax.set_title(SMOOTHING_LABELS[name], fontsize=9)
        ax.set_xlabel(r'$\theta_1$ [m/s]')
        plt.colorbar(cf, ax=ax, fraction=0.046)

    axes[0].set_ylabel(r'$\theta_2$ [m/s]')
    axes[0].legend(fontsize=7)
    fig.suptitle(r'Loss landscape $L_\kappa(\theta)$, $\kappa={}$'.format(
        int(kappa)), fontsize=11)
    fig.tight_layout()
    path = f'{OUT}/loss_landscapes.pdf'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')


# ============================================================
# Experiment 3: Gradient quality -- FD vs FoG
# ============================================================

def plot_gradient_comparison(kappa=KAPPA, n=15):
    """
    Compare FD and FoG gradient fields for each smoothing function.
    Also show the norm of (FoG - FD) as a measure of gradient error.
    """
    theta_star = analytical_optimum(PARAMS)

    fig, axes = plt.subplots(2, len(COMPARE),
                             figsize=(14, 7))

    T1, T2 = theta_grid(n=n, lo=-2.5, hi=2.5)

    for col_idx, name in enumerate(COMPARE):
        sigma_fn, sigma_prime_fn = get_fn(name)

        G_fd  = np.zeros((*T1.shape, 2))
        G_fog = np.zeros((*T1.shape, 2))
        err   = np.zeros(T1.shape)

        for i in range(n):
            for j in range(n):
                theta = np.array([T1[i, j], T2[i, j]])
                gfd  = grad_fd(theta, sigma_fn, kappa, PARAMS)
                gfog = grad_fog(theta, sigma_fn, sigma_prime_fn,
                                kappa, PARAMS)
                G_fd[i, j]  = gfd
                G_fog[i, j] = gfog
                err[i, j]   = np.linalg.norm(gfog - gfd)

        # Row 0: FD gradient field
        ax0 = axes[0, col_idx]
        ax0.quiver(T1[::2, ::2], T2[::2, ::2],
                   G_fd[::2, ::2, 0], G_fd[::2, ::2, 1],
                   color=SMOOTHING_COLORS[name], alpha=0.8,
                   scale=None, width=0.005)
        ax0.scatter(*theta_star, c='red', s=60, zorder=5)
        ax0.set_title(f'FD grad\n{SMOOTHING_LABELS[name]}', fontsize=8)
        ax0.set_aspect('equal')

        # Row 1: FoG gradient error ||FoG - FD||
        ax1 = axes[1, col_idx]
        cf = ax1.contourf(T1, T2, err, levels=15, cmap='Reds')
        ax1.scatter(*theta_star, c='blue', s=60, zorder=5)
        ax1.set_title(r'$\|\nabla^{[1]}L - \nabla^{FD}L\|$', fontsize=8)
        plt.colorbar(cf, ax=ax1, fraction=0.046)
        ax1.set_aspect('equal')

        for ax in [ax0, ax1]:
            ax.set_xlabel(r'$\theta_1$')
    axes[0, 0].set_ylabel(r'$\theta_2$')
    axes[1, 0].set_ylabel(r'$\theta_2$')

    fig.suptitle(r'Gradient comparison (FD vs FoG), $\kappa={}$'.format(
        int(kappa)), fontsize=11)
    fig.tight_layout()
    path = f'{OUT}/gradient_comparison.pdf'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')


# ============================================================
# Experiment 4: Gradient descent convergence
# ============================================================

def gradient_descent(theta0, sigma_fn, sigma_prime_fn, kappa, params,
                     lr=0.05, n_iter=300, use_fog=True):
    """Run gradient descent and return trajectory of theta and loss."""
    theta = theta0.copy()
    hist_theta = [theta.copy()]
    hist_loss  = [loss(theta, sigma_fn, kappa, params)]

    for _ in range(n_iter):
        if use_fog:
            g = grad_fog(theta, sigma_fn, sigma_prime_fn, kappa, params)
        else:
            g = grad_fd(theta, sigma_fn, kappa, params)
        theta = theta - lr * g
        hist_theta.append(theta.copy())
        hist_loss.append(loss(theta, sigma_fn, kappa, params))

    return np.array(hist_theta), np.array(hist_loss)


def plot_convergence(kappa=KAPPA, n_iter=200, lr=0.03):
    theta_star = analytical_optimum(PARAMS)
    theta0_list = [
        np.array([0.5,  0.3]),
        np.array([-1.0, 1.5]),
        np.array([2.5, -0.5]),
    ]
    colors_init = ['#333', '#666', '#999']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for name in COMPARE:
        sigma_fn, sigma_prime_fn = get_fn(name)
        col = SMOOTHING_COLORS[name]
        lbl = SMOOTHING_LABELS[name]

        for k, theta0 in enumerate(theta0_list):
            hist_theta, hist_loss = gradient_descent(
                theta0, sigma_fn, sigma_prime_fn,
                kappa, PARAMS, lr=lr, n_iter=n_iter)

            # Loss curve
            axes[0].semilogy(hist_loss,
                             color=col, alpha=0.6,
                             lw=1.5 if k == 0 else 0.8,
                             label=lbl if k == 0 else None)

            # Distance to optimum
            dist = np.linalg.norm(hist_theta - theta_star[None, :], axis=1)
            axes[1].semilogy(dist,
                             color=col, alpha=0.6,
                             lw=1.5 if k == 0 else 0.8,
                             label=lbl if k == 0 else None)

    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel(r'$L_\kappa(\theta^{(n)})$')
    axes[0].set_title('Loss convergence')
    axes[0].legend(fontsize=8)

    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel(r'$\|\theta^{(n)} - \theta^*\|$')
    axes[1].set_title('Distance to analytical optimum')

    fig.suptitle(r'Gradient descent (FoG), $\kappa={}$, $\eta={}$'.format(
        int(kappa), lr), fontsize=11)
    fig.tight_layout()
    path = f'{OUT}/convergence.pdf'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')


# ============================================================
# Experiment 5: Smoothing bias vs kappa
# ============================================================

def find_numerical_minimum(sigma_fn, kappa, n=50):
    """Find the numerical minimum of L_kappa by grid search."""
    T1, T2 = theta_grid(n=n, lo=-3.5, hi=3.5)
    best_L = np.inf
    best_theta = np.zeros(2)
    for i in range(n):
        for j in range(n):
            theta = np.array([T1[i, j], T2[i, j]])
            L = loss(theta, sigma_fn, kappa, PARAMS)
            if L < best_L:
                best_L = L
                best_theta = theta.copy()
    return best_theta


def plot_smoothing_bias():
    """
    Plot ||theta*_kappa - theta*|| as a function of kappa
    for each smoothing function.
    """
    kappas = np.logspace(0, 3, 20)   # 1 to 1000
    theta_star = analytical_optimum(PARAMS)

    fig, ax = plt.subplots(figsize=(7, 4))

    for name in COMPARE:
        sigma_fn, _ = get_fn(name)
        biases = []
        for k in kappas:
            theta_k = find_numerical_minimum(sigma_fn, k, n=40)
            biases.append(np.linalg.norm(theta_k - theta_star))
        ax.loglog(kappas, biases,
                  color=SMOOTHING_COLORS[name],
                  ls=SMOOTHING_LS[name],
                  lw=2,
                  label=SMOOTHING_LABELS[name])

    ax.set_xlabel(r'Stiffness $\kappa$')
    ax.set_ylabel(r'Smoothing bias $\|\theta^*_\kappa - \theta^*\|$')
    ax.set_title('Smoothing bias as a function of stiffness')
    ax.legend(fontsize=8)
    ax.grid(True, which='both', alpha=0.3)
    fig.tight_layout()
    path = f'{OUT}/smoothing_bias.pdf'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')


# ============================================================
# Experiment 6: ZoG vs FoG at a single point
# ============================================================

def plot_zog_vs_fog(kappa=KAPPA, n_samples_list=None):
    """
    Compare ZoG and FoG estimates at a fixed theta.
    Shows how ZoG variance decreases with N, and how FoG
    compares to the FD ground truth.
    """
    if n_samples_list is None:
        n_samples_list = [10, 50, 100, 500, 1000]

    # Evaluate at a point where contact happens
    theta_eval = np.array([1.0, 0.5])

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    for name in COMPARE:
        sigma_fn, sigma_prime_fn = get_fn(name)
        col = SMOOTHING_COLORS[name]

        # FD ground truth
        gfd = grad_fd(theta_eval, sigma_fn, kappa, PARAMS)

        # FoG
        gfog = grad_fog(theta_eval, sigma_fn, sigma_prime_fn,
                        kappa, PARAMS)

        # ZoG at varying N
        rng = np.random.default_rng(42)
        zog_means   = []
        zog_stds    = []
        sigma_noise = 0.1

        for N in n_samples_list:
            estimates = []
            for _ in range(30):   # 30 runs for variance estimate
                g = grad_zog(theta_eval, sigma_fn, kappa, PARAMS,
                             sigma_noise=sigma_noise,
                             N_samples=N, rng=rng)
                estimates.append(g)
            estimates = np.array(estimates)
            zog_means.append(np.mean(estimates, axis=0))
            zog_stds.append(np.std(np.linalg.norm(estimates, axis=1)))

        zog_norms = [np.linalg.norm(m) for m in zog_means]

        axes[0].errorbar(n_samples_list, zog_norms,
                         yerr=zog_stds,
                         color=col, lw=1.5, capsize=3,
                         label=SMOOTHING_LABELS[name])
        axes[0].axhline(np.linalg.norm(gfd), color=col,
                        ls=':', lw=1.0)

    axes[0].set_xscale('log')
    axes[0].set_xlabel('Number of ZoG samples $N$')
    axes[0].set_ylabel(r'$\|\nabla^{[0]} F(\theta)\|$')
    axes[0].set_title('ZoG gradient norm vs sample size\n(dotted = FD reference)')
    axes[0].legend(fontsize=7)

    # Bar chart: FD vs FoG for each smoothing
    names_short = COMPARE
    fd_norms  = []
    fog_norms = []
    fog_errs  = []

    for name in names_short:
        sigma_fn, sigma_prime_fn = get_fn(name)
        gfd  = grad_fd(theta_eval, sigma_fn, kappa, PARAMS)
        gfog = grad_fog(theta_eval, sigma_fn, sigma_prime_fn,
                        kappa, PARAMS)
        fd_norms.append(np.linalg.norm(gfd))
        fog_norms.append(np.linalg.norm(gfog))
        fog_errs.append(np.linalg.norm(gfog - gfd))

    x = np.arange(len(names_short))
    w = 0.3
    axes[1].bar(x - w/2, fd_norms,  width=w, label='FD (reference)',
                color='#555', alpha=0.8)
    axes[1].bar(x + w/2, fog_norms, width=w, label='FoG (analytical)',
                color=[SMOOTHING_COLORS[n] for n in names_short],
                alpha=0.8)
    axes[1].bar(x + w/2, fog_errs,  width=w, label=r'$\|$FoG$-$FD$\|$',
                color='red', alpha=0.4, bottom=0)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(
        [SMOOTHING_LABELS[n].split('$')[0].strip() for n in names_short],
        fontsize=8)
    axes[1].set_ylabel('Gradient norm')
    axes[1].set_title(r'FD vs FoG at $\theta = (1.0, 0.5)$')
    axes[1].legend(fontsize=8)

    fig.suptitle(r'ZoG vs FoG comparison, $\kappa={}$'.format(int(kappa)),
                 fontsize=11)
    fig.tight_layout()
    path = f'{OUT}/zog_vs_fog.pdf'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')


# ============================================================
# Experiment 7: Gradient along trajectory to optimum
# ============================================================

def plot_gradient_along_path(kappa=KAPPA, lr=0.03, n_iter=150):
    """
    Show gradient norm and FoG/FD agreement along the GD trajectory.
    """
    theta0     = np.array([0.5, 0.3])
    theta_star = analytical_optimum(PARAMS)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    for name in COMPARE:
        sigma_fn, sigma_prime_fn = get_fn(name)
        col = SMOOTHING_COLORS[name]
        lbl = SMOOTHING_LABELS[name]

        theta = theta0.copy()
        fog_norms = []
        fd_norms  = []
        errors    = []

        for _ in range(n_iter):
            gfd  = grad_fd(theta, sigma_fn, kappa, PARAMS)
            gfog = grad_fog(theta, sigma_fn, sigma_prime_fn,
                            kappa, PARAMS)
            fog_norms.append(np.linalg.norm(gfog))
            fd_norms.append(np.linalg.norm(gfd))
            errors.append(np.linalg.norm(gfog - gfd))
            theta = theta - lr * gfog

        iters = np.arange(n_iter)
        axes[0].semilogy(iters, fog_norms, color=col, lw=1.5, label=lbl)
        axes[0].semilogy(iters, fd_norms,  color=col, lw=1.0, ls='--',
                         alpha=0.5)
        axes[1].semilogy(iters, errors,    color=col, lw=1.5, label=lbl)

    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Gradient norm')
    axes[0].set_title('FoG norm (solid) vs FD norm (dashed)')
    axes[0].legend(fontsize=8)

    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel(r'$\|\nabla^{[1]}L - \nabla^{FD}L\|$')
    axes[1].set_title('FoG error along GD trajectory')

    fig.suptitle(r'Gradient quality during optimisation, $\kappa={}$'.format(
        int(kappa)), fontsize=11)
    fig.tight_layout()
    path = f'{OUT}/gradient_along_path.pdf'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')


# ============================================================
# Main: run all experiments
# ============================================================

if __name__ == '__main__':
    print('='*60)
    print('Analytical optimum:')
    theta_star = analytical_optimum(PARAMS)
    t_c = PARAMS.t_c
    mug_tc = PARAMS.mu * PARAMS.g * t_c
    print(f'  t_c = {t_c:.4f} s')
    print(f'  mu*g*t_c = {mug_tc:.4f} m/s  (sticking threshold)')
    print(f'  theta* = {theta_star}')
    print(f'  ||theta*|| = {np.linalg.norm(theta_star):.4f} m/s')
    regime = 'STICKING' if np.linalg.norm(theta_star) <= mug_tc else 'SLIDING'
    print(f'  Regime: {regime}')
    print()

    print('Running Experiment 1: Smoothing functions ...')
    plot_smoothing_functions()

    print('Running Experiment 2: Loss landscapes ...')
    plot_loss_landscapes(n=30)

    print('Running Experiment 3: Gradient comparison ...')
    plot_gradient_comparison(n=12)

    print('Running Experiment 4: Convergence ...')
    plot_convergence(n_iter=150, lr=0.03)

    print('Running Experiment 5: Smoothing bias ...')
    plot_smoothing_bias()

    print('Running Experiment 6: ZoG vs FoG ...')
    plot_zog_vs_fog()

    print('Running Experiment 7: Gradient along path ...')
    plot_gradient_along_path()

    print()
    print('All experiments complete. Figures saved to:', OUT)

In [ ]:
"""
simulator.py -- Moreau time-stepping with always-in-contact smoothing.
"""
import numpy as np
from smoothing import get_smoothing

class Params:
    def __init__(self):
        self.m=1.0; self.g=9.81; self.dt=1e-3; self.mu=0.5
        self.e=0.0; self.T=1.0; self.z0=1.0
        self.xy_target=np.array([1.0,0.5]); self.z_target=0.0
    @property
    def N(self): return int(self.T/self.dt)
    @property
    def t_c(self): return np.sqrt(2*self.z0/self.g)

def prox_normal(p_N):
    return max(p_N, 0.0)

def prox_friction_disk(p_T, p_N, mu):
    r = mu*max(p_N, 0.0)
    nrm = np.linalg.norm(p_T)
    if nrm <= r: return p_T.copy()
    if nrm == 0.0: return np.zeros(2)
    return p_T*(r/nrm)

def prox_contact(p, mu):
    p_N_proj = prox_normal(p[2])
    p_T_proj = prox_friction_disk(p[:2], p_N_proj, mu)
    return np.array([p_T_proj[0], p_T_proj[1], p_N_proj])

def solve_contact_GS(G, c, mu, n_iter=20):
    try: p = -np.linalg.solve(G, c)
    except: p = -np.linalg.pinv(G)@c
    r = 1.0/(1.0+np.sum(np.abs(G)))
    for _ in range(n_iter):
        p = prox_contact(p - r*(G@p + c), mu)
    return p

def moreau_step_smooth(q_S, v_S, sigma_fn, kappa, params):
    m=params.m; g=params.g; dt=params.dt; mu=params.mu
    q_M = q_S + 0.5*dt*v_S
    d = -q_M[2]
    sigma_val = float(sigma_fn(d, kappa))
    v_free = v_S + dt*np.array([0.,0.,-g])
    G = (1./m)*np.eye(3)
    c = v_free.copy()
    p_raw = solve_contact_GS(G, c, mu)
    p_smooth = p_raw * sigma_val
    v_E = v_free + (1./m)*p_smooth
    q_E = q_M + 0.5*dt*v_E
    return q_E, v_E, p_raw, sigma_val, d

def simulate(theta, sigma_fn, kappa, params):
    q=np.array([0.,0.,params.z0]); v=np.array([theta[0],theta[1],0.])
    N=params.N
    traj_q=np.zeros((N+1,3)); traj_v=np.zeros((N+1,3))
    traj_p=np.zeros((N,3));   traj_s=np.zeros(N); traj_d=np.zeros(N)
    traj_q[0]=q; traj_v[0]=v
    for k in range(N):
        q,v,p_raw,s,d = moreau_step_smooth(q,v,sigma_fn,kappa,params)
        traj_q[k+1]=q; traj_v[k+1]=v; traj_p[k]=p_raw
        traj_s[k]=s;   traj_d[k]=d
    return traj_q, traj_v, traj_p, traj_s, traj_d

def loss(theta, sigma_fn, kappa, params):
    traj_q,_,_,_,_ = simulate(theta, sigma_fn, kappa, params)
    q_T = traj_q[-1]
    diff_xy = q_T[:2] - params.xy_target
    return float(diff_xy@diff_xy + (q_T[2]-params.z_target)**2)

def grad_fd(theta, sigma_fn, kappa, params, h=1e-5):
    g=np.zeros(2)
    for i in range(2):
        tp=theta.copy(); tp[i]+=h
        tm=theta.copy(); tm[i]-=h
        g[i]=(loss(tp,sigma_fn,kappa,params)-loss(tm,sigma_fn,kappa,params))/(2*h)
    return g

def grad_fog(theta, sigma_fn, sigma_prime_fn, kappa, params):
    m=params.m; g=params.g; dt=params.dt; mu=params.mu; N=params.N

    # Forward pass
    q=np.array([0.,0.,params.z0]); v=np.array([theta[0],theta[1],0.])
    states_q=[q.copy()]; states_v=[v.copy()]
    store_p=[]; store_s=[]; store_sp=[]; store_d=[]

    for k in range(N):
        q_M = q+0.5*dt*v; d=-q_M[2]
        s=float(sigma_fn(d,kappa)); sp=float(sigma_prime_fn(d,kappa))
        v_free=v+dt*np.array([0.,0.,-g])
        G=(1./m)*np.eye(3); c=v_free.copy()
        p_raw=solve_contact_GS(G,c,mu)
        v=v_free+(1./m)*p_raw*s
        q=q_M+0.5*dt*v
        states_q.append(q.copy()); states_v.append(v.copy())
        store_p.append(p_raw.copy()); store_s.append(s)
        store_sp.append(sp); store_d.append(d)

    # Loss gradient w.r.t. final state
    q_T=states_q[-1]
    dL_dqT=np.zeros(3)
    dL_dqT[:2]=2.*(q_T[:2]-params.xy_target)
    dL_dqT[2]=2.*(q_T[2]-params.z_target)

    # Propagate Jacobians J_q=dq_k/dtheta (3x2), J_v=dv_k/dtheta (3x2)
    J_q=np.zeros((3,2)); J_v=np.zeros((3,2))
    J_v[0,0]=1.; J_v[1,1]=1.

    for k in range(N):
        p_raw=store_p[k]; s=store_s[k]; sp=store_sp[k]
        p_N=p_raw[2]; p_T=p_raw[:2]; nrm_T=np.linalg.norm(p_T)
        mu_pN=mu*p_N

        # Jacobian of prox_N
        J_pN_xN = 1.0 if p_N > 0 else 0.0

        # Jacobian of prox_T
        if nrm_T <= mu_pN:
            J_pT_xT=np.eye(2); J_pT_xN=np.zeros(2)
        elif nrm_T > 0:
            e_T=p_T/nrm_T
            J_pT_xT=(mu_pN/nrm_T)*(np.eye(2)-np.outer(e_T,e_T))
            J_pT_xN=(mu/nrm_T)*p_T*J_pN_xN
        else:
            J_pT_xT=np.zeros((2,2)); J_pT_xN=np.zeros(2)

        # dp_raw/dv_free (3x3): x = -m*v_free
        J_praw_vfree=np.zeros((3,3))
        J_praw_vfree[2,2]=J_pN_xN*(-m)
        J_praw_vfree[:2,:2]=J_pT_xT*(-m)
        J_praw_vfree[:2,2]=J_pT_xN*(-m)

        # dd/d(q_S, v_S): d=-q_z^M=-q_z^S-dt/2*v_z^S
        dd_dqS=np.zeros(3); dd_dqS[2]=-1.
        dd_dvS=np.zeros(3); dd_dvS[2]=-dt/2.

        # dp_smooth/d(q_S,v_S) = sigma*dp_raw/dv_free@dv_free/d(q_S,v_S)
        #                       + sigma'*outer(p_raw, dd/d(q_S,v_S))
        # dv_free/dq_S=0, dv_free/dv_S=I
        J_psmooth_qS = sp*np.outer(p_raw, dd_dqS)
        J_psmooth_vS = s*J_praw_vfree + sp*np.outer(p_raw, dd_dvS)

        # dv_E/d(q_S,v_S)
        J_vE_qS=(1./m)*J_psmooth_qS
        J_vE_vS=np.eye(3)+(1./m)*J_psmooth_vS

        # dq_E/d(q_S,v_S): q_E=q_M+dt/2*v_E, q_M=q_S+dt/2*v_S
        J_qE_qS=np.eye(3)+0.5*dt*J_vE_qS
        J_qE_vS=0.5*dt*np.eye(3)+0.5*dt*J_vE_vS

        # Chain rule
        J_q_new=J_qE_qS@J_q+J_qE_vS@J_v
        J_v_new=J_vE_qS@J_q+J_vE_vS@J_v
        J_q=J_q_new; J_v=J_v_new

    return dL_dqT@J_q

def grad_zog(theta, sigma_fn, kappa, params, sigma_noise=0.1,
             N_samples=100, rng=None):
    if rng is None: rng=np.random.default_rng(0)
    W=rng.normal(0.,sigma_noise,size=(N_samples,2))
    grad=np.zeros(2)
    for i in range(N_samples):
        w=W[i]; L_i=loss(theta+w, sigma_fn, kappa, params)
        grad+=(w/sigma_noise**2)*L_i
    return grad/N_samples

def analytical_optimum(params):
    t_c=params.t_c; mu=params.mu; g=params.g
    q_st=params.xy_target; d_st=np.linalg.norm(q_st)
    direction=q_st/d_st
    theta_sticking=d_st/t_c
    if theta_sticking <= mu*g*t_c:
        return direction*theta_sticking
    d_star2=d_st-0.5*mu*g*t_c**2
    a=1./(2.*mu*g); b=t_c; c_coef=-d_star2
    disc=b**2-4.*a*c_coef
    if disc < 0: return direction*theta_sticking
    u=(-b+np.sqrt(disc))/(2.*a)
    return direction*(u+mu*g*t_c)


In [ ]:
import numpy as np
from scipy.special import erf as _scipy_erf

def sigmoid(d, kappa):
    x = np.asarray(d, dtype=float) * kappa
    return np.where(x >= 0, 1.0/(1.0+np.exp(-x)), np.exp(x)/(1.0+np.exp(x)))

def sigmoid_prime(d, kappa):
    s = sigmoid(d, kappa)
    return kappa * s * (1.0 - s)

def erf_smooth(d, kappa):
    return 0.5*(1.0 + _scipy_erf(np.asarray(d,dtype=float)*kappa/np.sqrt(2.0)))

def erf_smooth_prime(d, kappa):
    return (kappa/np.sqrt(2.0*np.pi))*np.exp(-0.5*(np.asarray(d,dtype=float)*kappa)**2)

def smoothstep(d, kappa):
    s = np.clip(0.5 + kappa*np.asarray(d,dtype=float), 0.0, 1.0)
    return 3.0*s**2 - 2.0*s**3

def smoothstep_prime(d, kappa):
    d = np.asarray(d, dtype=float)
    s_raw = 0.5 + kappa*d
    in_support = (s_raw > 0.0) & (s_raw < 1.0)
    s = np.clip(s_raw, 0.0, 1.0)
    return np.where(in_support, 6.0*kappa*s*(1.0-s), 0.0)

def sigmoid_mass(d, kappa, mass):
    return sigmoid(d, kappa*mass)

def sigmoid_mass_prime(d, kappa, mass):
    return sigmoid_prime(d, kappa*mass)

def hard(d, kappa):
    return (np.asarray(d,dtype=float) >= 0).astype(float)

def hard_prime(d, kappa):
    return np.zeros_like(np.asarray(d,dtype=float))

def get_smoothing(name, mass=1.0):
    if name == 'sigmoid':
        return sigmoid, sigmoid_prime
    elif name == 'erf':
        return erf_smooth, erf_smooth_prime
    elif name == 'smoothstep':
        return smoothstep, smoothstep_prime
    elif name == 'sigmoid_mass':
        return (lambda d,k: sigmoid_mass(d,k,mass),
                lambda d,k: sigmoid_mass_prime(d,k,mass))
    elif name == 'hard':
        return hard, hard_prime
    else:
        raise ValueError(f"Unknown smoothing: {name}")

SMOOTHING_NAMES  = ['sigmoid','erf','smoothstep','sigmoid_mass','hard']
SMOOTHING_LABELS = {
    'sigmoid':      r'Sigmoid $\sigma_{\rm sig}$',
    'erf':          r'Erf (Gaussian CDF) $\sigma_{\rm erf}$',
    'smoothstep':   r'Smoothstep $\sigma_{\rm poly}$',
    'sigmoid_mass': r'Mass-scaled $\sigma_{\rm mass}$',
    'hard':         r'Hard contact',
}
SMOOTHING_COLORS = {
    'sigmoid':'#2196F3','erf':'#E91E63','smoothstep':'#4CAF50',
    'sigmoid_mass':'#FF9800','hard':'#9E9E9E',
}
SMOOTHING_LS = {
    'sigmoid':'-','erf':'--','smoothstep':'-.','sigmoid_mass':':',
    'hard':(0,(3,1,1,1)),
}
